# Data Transformation by Rishab Kumar
## Here the bronze data is 
### - transformed to a high quality data
### - prepared as a part of Silver layer

In [0]:
%sql
use catalog dbacademy;
use schema dbuser_100426;


In [0]:
menu_df_1 = spark.table('bronze_menu_items_csv_file')
menu_df_2 = spark.table('bronze_menu_items_parquet_file')
restaurant_df_1 = spark.table('bronze_restaurant_csv_file')
restaurant_df_2 = spark.table('bronze_restaurant_parquet_file')

In [0]:

menu_schema_1 = set((f.name, f.dataType) for f in menu_df_1.schema.fields)
menu_schema_2 = set((f.name, f.dataType) for f in menu_df_2.schema.fields)
restaurant_schema_1 = set((f.name, f.dataType) for f in restaurant_df_1.schema.fields)
restaurant_schema_2 = set((f.name, f.dataType) for f in restaurant_df_2.schema.fields)
print("columns only in menu_df_1: ", menu_schema_1 - menu_schema_2)
print("--------------------")
print("columns only in menu_df_2: ", menu_schema_2 - menu_schema_1)
print("--------------------")
print("columns only in restaurant_df_1: ", restaurant_schema_1 - restaurant_schema_2)
print("--------------------")
print("columns only in restaurant_df_2: ", restaurant_schema_2 - restaurant_schema_1)
print("--------------------")
print("------ SCHEMA ------")
print("menu_df_1")
menu_df_1.printSchema()
print("menu_df_2")
menu_df_2.printSchema()
print("restaurant_df_1")
restaurant_df_1.printSchema()
print("restaurant_df_2")
restaurant_df_2.printSchema()


columns only in menu_df_1:  {('price', StringType())}
--------------------
columns only in menu_df_2:  {('price', FloatType())}
--------------------
columns only in restaurant_df_1:  {('latitude', StringType()), ('avg_rating', StringType()), ('total_ratings', StringType()), ('cost_for_two', StringType())}
--------------------
columns only in restaurant_df_2:  {('latitude', DoubleType()), ('total_ratings', IntegerType()), ('avg_rating', FloatType()), ('cost_for_two', IntegerType())}
--------------------
------ SCHEMA ------
menu_df_1
root
 |-- restaurant_id: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: string (nullable = true)
 |-- description: string (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- r

### The columns and datatypes to be implemented as mentioned below

In [0]:
# enforcing the datatypes as follows

# menu_items.csv / menu_items.parquet

# Column	            Type	  Description
# restaurant_id	      string	Foreign key to restaurants table
# restaurant_name	    string	Restaurant name
# city_name	          string	City name
# city_slug	          string	URL-friendly city identifier
# category	          string	Menu category (e.g. Recommended, Biryani, Starters > Non Veg)
# item_id	            string	Unique Swiggy menu item identifier
# item_name	          string	Name of the menu item
# price	              float	  Price in INR
# description	        string	Item description provided by restaurant
# is_veg	            bool	  Whether the item is vegetarian
# rating	            string	Average item rating (if available)
# rating_count	      string	Number of ratings for this item
# in_stock	          bool	  Whether the item was in stock at time of scraping
# has_addons	        bool	  Whether the item has addon options (sauces, sides, drinks, etc.)
# has_variants	      bool	  Whether the item has size/variant options



# restaurants.csv / restaurants.parquet

# restaurant_id	      string	Unique Swiggy restaurant identifier
# name	              string	Restaurant name
# area	              string	Area / neighborhood name
# locality	          string	Specific locality within the area
# city_name	          string	City name
# city_slug	          string	URL-friendly city identifier
# latitude	          float	  City center latitude
# longitude	          float	  City center longitude
# cuisines	          string	Pipe-separated list of cuisine types (e.g. Indian|Chinese|Fast Food)
# avg_rating	        float	  Average customer rating (out of 5.0)
# total_ratings	      int	    Total number of ratings received
# cost_for_two	      int	    Approximate cost for two people in INR
# delivery_time_mins	int	    Estimated delivery time in minutes
# is_veg	            bool	  Whether the restaurant is pure vegetarian
# is_open	            bool	  Whether the restaurant was open at time of scraping
# menu_category_count	int	    Number of menu categories
# menu_item_count	    int	    Total number of menu items




In [0]:
# menu_df_1.columns
# menu_df_2.columns
# restaurant_df_1.columns
# restaurant_df_2.columns


### Defining the functions 

In [0]:
import pyspark.sql.functions as F


# a function that takes a dataframe and a name and prints the total number of records and the unique counts of each column
def get_report_for_unique_values(df, name):
    output={}
    print(f"Total records: {df.count()}")
    print(f"Unique counts of each column in {name}")
    for col in df.columns:
        output[col] = df.select(col).distinct().count()
    return output

# a function that takes a dataframe and a column name and prints the average number of records for each unique value in a column
def density_analysis(df, column_name):
    return df.count()/df.select(column_name).distinct().count()

def safe_exec(func):
    try:
        return func()
    except:
        return None

# a function that takes a dataframe and a column name and returns the stats
def get_column_stats(df, column):
    output = {}
    output["distinct"] = safe_exec(lambda: df.select(column).distinct().count())
    output["null"]     = safe_exec(lambda: df.filter(F.col(column).isNull()).count())
    output["max"]      = safe_exec(lambda: df.select(F.max(column)).collect()[0][0])
    output["min"]      = safe_exec(lambda: df.select(F.min(column)).collect()[0][0])
    output["avg"]      = safe_exec(lambda: df.select(F.avg(column)).collect()[0][0])
    return output

# a function that takes a dataframe and a column name and converts it to float datatype
def float_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned", F.regexp_replace(column, "[^0-9.]", "")
    )
    df = df.withColumn(
        column + "_float", F.trim(F.col(column + "_cleaned")).try_cast("decimal(10,2)") 
    )
    df = df.drop(column + "_cleaned")
    df = df.drop(column).withColumnRenamed(column + "_float", column)
    return df

def bool_conversion(df, column):
    df = df.withColumn(column + "_cleaned", F.lower(F.regexp_replace(column, "[^a-z0-9]", "")))
    df = df.withColumn(column + "_bool", 
                       F.when(F.col(column + "_cleaned").isin("true", "yes", "y", "1"), True)
                       .when(F.col(column + "_cleaned").isin("false", "no", "n", "0"), False)
                       .otherwise(F.lit(None)))
    df = df.drop(column + "_cleaned")
    df = df.drop(column).withColumnRenamed(column + "_bool", column)
    return df

# a function that takes a dataframe and a column name and converts it to int datatype
def int_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned", F.regexp_replace(column, "[^0-9]", "")
    )
    df = df.withColumn(
        column + "_int", F.trim(F.col(column + "_cleaned")).try_cast("decimal(20,0)") 
    )
    df = df.drop(column + "_cleaned")
    df = df.drop(column).withColumnRenamed(column + "_int", column)
    return df

# a function that takes a dataframe and a column name and cleans it to id
def id_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned_id", 
        F.when(F.col(column).rlike("^[0-9]+$"), F.col(column))
    )
    df = df.drop(column).withColumnRenamed(column + "_cleaned_id", column)
    df = df.filter(F.col(column).isNotNull())
    return df

# a function that takes a dataframe and a latitude and longitude and converts it to float datatype
def conversion_for_latitude_longitude(df, lat, lon):
    #latitude
    df = df.withColumn(
        lat + "_cleaned", F.regexp_replace(lat, "[^0-9.-]", "")
    )
    df = df.withColumn(
        lat + "_float", F.trim(F.col(lat + "_cleaned")).try_cast("decimal") 
    )
    df = df.drop(lat + "_cleaned")
    df = df.drop(lat).withColumnRenamed(lat + "_float", lat)

    #longitude
    df = df.withColumn(
        lon + "_cleaned", F.regexp_replace(lon, "[^0-9.-]", "")
    )
    df = df.withColumn(
        lon + "_float", F.trim(F.col(lon + "_cleaned")).try_cast("decimal") 
    )
    df = df.drop(lon + "_cleaned")
    df = df.drop(lon).withColumnRenamed(lon + "_float", lon)
    return df

def cuisines_to_array(df, column):
    df = df.withColumn(column + "_list", 
                       F.transform(
                           F.array_distinct(F.split(F.lower(F.col(column)), "|"))), 
                       lambda x: F.trim(x))
    return df

def replace_exceptional_char(df, column):
    df = df.withColumn(column, F.regexp_replace(column, "[^a-z0-9 &'-]", ""))
    return df

# a function that maintaines the lowercase of the columns
def maintain_lower_case(df, columns):
    for column in columns:
        df = df.withColumn(column, F.lower(F.trim(F.col(column))))
    return df

In [0]:
# get_report_for_unique_values(menu_df_1, "menu_df_1")

## Working on "restaurant" Dataset

In [0]:
"""
restaurant_df_1
root
 |-- restaurant_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- area: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- avg_rating: string (nullable = true)
 |-- total_ratings: string (nullable = true)
 |-- cost_for_two: string (nullable = true)
 |-- delivery_time_mins: integer (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- is_open: string (nullable = true)
 |-- menu_category_count: integer (nullable = true)
 |-- menu_item_count: integer (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

 restaurant_df_2
root
 |-- restaurant_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- area: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- avg_rating: float (nullable = true)
 |-- total_ratings: integer (nullable = true)
 |-- cost_for_two: integer (nullable = true)
 |-- delivery_time_mins: integer (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- is_open: string (nullable = true)
 |-- menu_category_count: integer (nullable = true)
 |-- menu_item_count: integer (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

"""

'\nrestaurant_df_1\nroot\n |-- restaurant_id: string (nullable = true)\n |-- name: string (nullable = true)\n |-- area: string (nullable = true)\n |-- locality: string (nullable = true)\n |-- city_name: string (nullable = true)\n |-- city_slug: string (nullable = true)\n |-- latitude: string (nullable = true)\n |-- longitude: double (nullable = true)\n |-- cuisines: string (nullable = true)\n |-- avg_rating: string (nullable = true)\n |-- total_ratings: string (nullable = true)\n |-- cost_for_two: string (nullable = true)\n |-- delivery_time_mins: integer (nullable = true)\n |-- is_veg: string (nullable = true)\n |-- is_open: string (nullable = true)\n |-- menu_category_count: integer (nullable = true)\n |-- menu_item_count: integer (nullable = true)\n |-- ingestion_time: timestamp (nullable = true)\n\n restaurant_df_2\nroot\n |-- restaurant_id: string (nullable = true)\n |-- name: string (nullable = true)\n |-- area: string (nullable = true)\n |-- locality: string (nullable = true)\n 

In [0]:
display(restaurant_df_1.limit(5))

restaurant_id,name,area,locality,city_name,city_slug,latitude,longitude,cuisines,avg_rating,total_ratings,cost_for_two,delivery_time_mins,is_veg,is_open,menu_category_count,menu_item_count,ingestion_time
1046859,KFC,MALOUT ROAD,VILLAGE GOBINDGARH,Abohar,abohar,30.15227909649705,74.19491876329153,Burgers|Fast Food,4.3,169,400,51,False,True,20,197,2026-04-12T18:12:06.570Z
171675,Domino's Pizza,Abohar Locality,Hanuman Garh Road,Abohar,abohar,30.15227909649705,74.19491876329153,Pizzas|Italian,4.2,1900,400,25,False,True,22,218,2026-04-12T18:12:06.570Z
1354956,Rominus Pizza and Burger,Central Abohar,Hanumangarh Road,Abohar,abohar,30.15227909649705,74.19491876329153,American|Pizzas,4.3,39,300,24,False,True,25,125,2026-04-12T18:12:06.570Z
1316616,Kundi Sotta,Central Abohar,Central Abohar,Abohar,abohar,30.15227909649705,74.19491876329153,North Indian|Indian,4.1,8,300,27,True,True,2,25,2026-04-12T18:12:06.570Z
1067362,Smokinn Pizzeria,Central Abohar,Central Abohar,Abohar,abohar,30.15227909649705,74.19491876329153,Desserts|Pizzas,3.8,382,149,29,True,True,17,100,2026-04-12T18:12:06.570Z


In [0]:
clean_restaurant_df_1 = restaurant_df_1.drop_duplicates(subset=["restaurant_id"])
clean_restaurant_df_1 = id_conversion(clean_restaurant_df_1, "restaurant_id")
clean_restaurant_df_2 = restaurant_df_2.drop_duplicates(subset=["restaurant_id"])
clean_restaurant_df_2 = id_conversion(clean_restaurant_df_2, "restaurant_id")

# get stats for restaurant_id
# restaurant_id_stats = get_column_stats(clean_restaurant_df_1, "restaurant_id")
# print()
# print("------ restaurant_id_stats ------")
# display(restaurant_id_stats)


clean_restaurant_df_1 = conversion_for_latitude_longitude(clean_restaurant_df_1, "latitude", "longitude")
clean_restaurant_df_2 = conversion_for_latitude_longitude(clean_restaurant_df_2, "latitude", "longitude")
# get stats for latitude
# latitude_stats = get_column_stats(clean_restaurant_df_1, "latitude")
# print()
# print("------ latitude_stats ------")
# display(latitude_stats)
# get stats for longitude
# longitude_stats = get_column_stats(clean_restaurant_df_1, "longitude")
# print()
# print("------ longitude_stats ------")
# display(longitude_stats)


### Handling - "avg_rating"
clean_restaurant_df_1 = float_conversion(clean_restaurant_df_1, "avg_rating")
clean_restaurant_df_2 = float_conversion(clean_restaurant_df_2, "avg_rating")
# get stats for avg_rating
# avg_rating_stats = get_column_stats(clean_restaurant_df_1, "avg_rating")
# print()
# print("------ avg_rating_stats ------")
# display(avg_rating_stats)


### Handling - "total_ratings"
clean_restaurant_df_1 = int_conversion(clean_restaurant_df_1, "total_ratings")
clean_restaurant_df_2 = int_conversion(clean_restaurant_df_2, "total_ratings")
# get stats for total_ratings
# total_ratings_stats = get_column_stats(clean_restaurant_df_1, "total_ratings")
# print()
# print("------ total_ratings_stats ------")
# display(total_ratings_stats)


### Handling - "cost_for_two"
clean_restaurant_df_1 = int_conversion(clean_restaurant_df_1, "cost_for_two")
clean_restaurant_df_2 = int_conversion(clean_restaurant_df_2, "cost_for_two")
# get stats for cost_for_two
# cost_for_two_stats = get_column_stats(clean_restaurant_df_1, "cost_for_two")
# print()
# print("------ cost_for_two_stats ------")
# display(cost_for_two_stats)


#### Handling - "delivery_time_mins"
clean_restaurant_df_1 = int_conversion(clean_restaurant_df_1, "delivery_time_mins")
clean_restaurant_df_2 = int_conversion(clean_restaurant_df_2, "delivery_time_mins")
# get stats for delivery_time_mins
# delivery_time_mins_stats = get_column_stats(clean_restaurant_df_1, "delivery_time_mins")
# print()
# print("------ delivery_time_mins_stats ------")
# display(delivery_time_mins_stats)


### Handling - "is_veg" - "is_open"
clean_restaurant_df_1 = bool_conversion(clean_restaurant_df_1, "is_veg")
clean_restaurant_df_1 = bool_conversion(clean_restaurant_df_1, "is_open")
clean_restaurant_df_2 = bool_conversion(clean_restaurant_df_2, "is_veg")
clean_restaurant_df_2 = bool_conversion(clean_restaurant_df_2, "is_open")
# get stats for is_veg
# is_veg_stats = get_column_stats(clean_restaurant_df_1, "is_veg")
# print()
# print("------ is_veg_stats -----")
# display(is_veg_stats)
# get stats for is_open
# is_open_stats = get_column_stats(clean_restaurant_df_1, "is_open")
# print()
# print("------ is_open_stats -----")
# display(is_open_stats)


### Handling - "menu_category_count"
clean_restaurant_df_1 = int_conversion(clean_restaurant_df_1, "menu_category_count")
clean_restaurant_df_2 = int_conversion(clean_restaurant_df_2, "menu_category_count")
# get stats for menu_category_count
# menu_category_count_stats = get_column_stats(clean_restaurant_df_1, "menu_category_count")
# print()
# print("------ menu_category_count_stats -----")
# display(menu_category_count_stats)


### Handling - "menu_item_count"
clean_restaurant_df_1 = int_conversion(clean_restaurant_df_1, "menu_item_count")
clean_restaurant_df_2 = int_conversion(clean_restaurant_df_2, "menu_item_count")
# get stats for menu_item_count
# menu_item_count_stats = get_column_stats(clean_restaurant_df_1, "menu_item_count")
# print()
# print("------ menu_item_count_stats -----")
# display(menu_item_count_stats)
# cleaning the data with a subset of restaurant_id and item_id


clean_restaurant_df_1 = maintain_lower_case(clean_restaurant_df_1, 
                                            ["name", "area", "locality", "city_name", "city_slug", "cuisines"])
clean_restaurant_df_2 = maintain_lower_case(clean_restaurant_df_2, 
                                            ["name", "area", "locality", "city_name", "city_slug", "cuisines"])


## Working on "Menu" Dataset

### Handling duplicates -
### "restaurant_id" and "item_id" 

In [0]:
"""
menu_df_1
root
 |-- restaurant_id: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: string (nullable = true)
 |-- description: string (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- in_stock: string (nullable = true)
 |-- has_addons: string (nullable = true)
 |-- has_variants: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

menu_df_2
root
 |-- restaurant_id: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: float (nullable = true)
 |-- description: string (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- in_stock: string (nullable = true)
 |-- has_addons: string (nullable = true)
 |-- has_variants: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

"""

'\nmenu_df_1\nroot\n |-- restaurant_id: string (nullable = true)\n |-- restaurant_name: string (nullable = true)\n |-- city_name: string (nullable = true)\n |-- city_slug: string (nullable = true)\n |-- category: string (nullable = true)\n |-- item_id: string (nullable = true)\n |-- item_name: string (nullable = true)\n |-- price: string (nullable = true)\n |-- description: string (nullable = true)\n |-- is_veg: string (nullable = true)\n |-- rating: string (nullable = true)\n |-- rating_count: string (nullable = true)\n |-- in_stock: string (nullable = true)\n |-- has_addons: string (nullable = true)\n |-- has_variants: string (nullable = true)\n |-- ingestion_time: timestamp (nullable = true)\n\nmenu_df_2\nroot\n |-- restaurant_id: string (nullable = true)\n |-- restaurant_name: string (nullable = true)\n |-- city_name: string (nullable = true)\n |-- city_slug: string (nullable = true)\n |-- category: string (nullable = true)\n |-- item_id: string (nullable = true)\n |-- item_name: s

In [0]:
# cleaning the data with a subset of restaurant_id and item_id
clean_menu_df_1 = menu_df_1.drop_duplicates(subset=["restaurant_id","item_id"])
clean_menu_df_2 = menu_df_2.drop_duplicates(subset=["restaurant_id","item_id"])


clean_menu_df_1 = id_conversion(clean_menu_df_1, "restaurant_id")
clean_menu_df_2 = id_conversion(clean_menu_df_2, "restaurant_id")
#get stats for restaurant_id
# menu_restaurant_id_stats_1 = get_column_stats(clean_menu_df_1, "restaurant_id")
# print()
# print("------ menu_restaurant_id_stats_1 ------")
# display(menu_restaurant_id_stats_1)



clean_menu_df_1 = id_conversion(clean_menu_df_1, "item_id")
clean_menu_df_2 = id_conversion(clean_menu_df_2, "item_id")
#get stats for item_id
# menu_item_id_stats_1 = get_column_stats(clean_menu_df_1, "item_id")
# print()
# print("------ menu_item_id_stats_1 ------")
# display(menu_item_id_stats_1)


# convert price columns to float datatype
clean_menu_df_1 = float_conversion(clean_menu_df_1, "price").filter("price_float > 1")
clean_menu_df_2 = float_conversion(clean_menu_df_2, "price").filter("price_float > 1")
#get stats for price
# menu_price_stats_1 = get_column_stats(clean_menu_df_1, "price")
# print()
# print("------ menu_price_stats_1 ------")
# display(menu_price_stats_1)



clean_menu_df_1 = float_conversion(clean_menu_df_1, "rating")
clean_menu_df_2 = float_conversion(clean_menu_df_2, "rating")
#get stats for rating
# menu_rating_stats_1 = get_column_stats(clean_menu_df_1, "rating")
# print()
# print("------ menu_rating_stats_1 ------")
# display(menu_rating_stats_1)


clean_menu_df_1 = int_conversion(clean_menu_df_1, "rating_count")
clean_menu_df_2 = int_conversion(clean_menu_df_2, "rating_count")
# get stats for rating_count
# menu_rating_count_stats_1 = get_column_stats(clean_menu_df_1, "rating_count")
# print()
# print("------ menu_rating_count_stats_1 ------")
# display(menu_rating_count_stats_1)



list_columns = ["city_name", "city_slug", "category", "item_name", "description", "restaurant_name"]

clean_menu_df_1 = maintain_lower_case(clean_menu_df_1, list_columns)
clean_menu_df_2 = maintain_lower_case(clean_menu_df_2, list_columns)


clean_menu_df_1 = bool_conversion(clean_menu_df_1, "is_veg")
clean_menu_df_2 = bool_conversion(clean_menu_df_2, "is_veg")
# get stats for is_veg
# menu_is_veg_stats_1 = get_column_stats(clean_menu_df_1, "is_veg")
# print()
# print("------ menu_is_veg_stats_1 ------")
# display(menu_is_veg_stats_1)


clean_menu_df_1 = bool_conversion(clean_menu_df_1, "in_stock")
clean_menu_df_2 = bool_conversion(clean_menu_df_2, "in_stock")
# get stats for in_stock
# menu_in_stock_stats_1 = get_column_stats(clean_menu_df_1, "in_stock")
# print()
# print("------ menu_in_stock_stats_1 ------")
# display(menu_in_stock_stats_1)


clean_menu_df_1 = bool_conversion(clean_menu_df_1, "has_addons")
clean_menu_df_2 = bool_conversion(clean_menu_df_2, "has_addons")
# get stats for has_addons
# menu_has_addons_stats_1 = get_column_stats(clean_menu_df_1, "has_addons")
# print()
# print("------ menu_has_addons_stats_1 ------")
# display(menu_has_addons_stats_1)


clean_menu_df_1 = bool_conversion(clean_menu_df_1, "has_variants")
clean_menu_df_2 = bool_conversion(clean_menu_df_2, "has_variants") 
# get stats for has_variants
# menu_has_variants_stats_1 = get_column_stats(clean_menu_df_1, "has_variants")
# print()
# print("------ menu_has_variants_stats_1 ------")
# display(menu_has_variants_stats_1)



### Reordering the Dataframe

In [0]:

menu_ordered_columns = [
    "restaurant_id",	    
    "restaurant_name",	
    "city_name",   
    "city_slug",  
    "category", 
    "item_id",
    "item_name",
    "price",
    "description",
    "is_veg",
    "rating",
    "rating_count",
    "in_stock",
    "has_addons",
    "has_variants"]

restaurant_ordered_columns = [
    "restaurant_id",
    "name",
    "area",
    "locality",
    "city_name",
    "city_slug",
    "latitude",
    "longitude",
    "cuisines",
    "avg_rating",
    "total_ratings",
    "cost_for_two",
    "delivery_time_mins",
    "is_veg",
    "is_open",
    "menu_category_count",
    "menu_item_count"]


clean_menu_df_1 = clean_menu_df_1.select(menu_ordered_columns)
clean_menu_df_2 = clean_menu_df_2.select(menu_ordered_columns)
clean_restaurant_df_2 = clean_restaurant_df_2.select(restaurant_ordered_columns)
clean_restaurant_df_1 = clean_restaurant_df_1.select(restaurant_ordered_columns)
    

### performing merge operation on the two dataframes

In [0]:


#adding source column
clean_menu_df_1 = clean_menu_df_1.withColumn("source", F.lit("csv"))
clean_menu_df_2 = clean_menu_df_2.withColumn("source", F.lit("parquet"))
clean_restaurant_df_1 = clean_restaurant_df_1.withColumn("source", F.lit("csv"))
clean_restaurant_df_2 = clean_restaurant_df_2.withColumn("source", F.lit("parquet"))

#merging the two dataframes
clean_menu_df_1 = clean_menu_df_1.unionByName(clean_menu_df_2)
clean_restaurant_df_1 = clean_restaurant_df_1.unionByName(clean_restaurant_df_2)

#dropping duplicates
clean_menu_df_merged = clean_menu_df_1.dropDuplicates(["restaurant_id", "item_id"])
clean_restaurant_df_merged = clean_restaurant_df_1.dropDuplicates(["restaurant_id"])


In [0]:
# complete_report_clean_menu_df = get_report_for_unique_values(clean_menu_df_merged, "clean_menu_df_merged")
# complete_report_clean_restaurant_df = get_report_for_unique_values(clean_restaurant_df_merged,"clean_restaurant_df_1")
# display(complete_report_clean_menu_df)
# display(complete_report_clean_restaurant_df)

In [0]:

# saving at silver delta tables
clean_menu_df_merged.write.format("delta").mode("overwrite").saveAsTable("silver_menu_items_table")
clean_restaurant_df_merged.write.format("delta").mode("overwrite").saveAsTable("silver_restaurants_table")
